# Evaluasi Klasifikasi Multilabel dengan Label Tidak Seimbang

**Buku:** *Deep Learning dengan Python* — Bab 16, Mengevaluasi Model dengan Jujur

Setiap gambar mozaik 2×2 berisi beberapa jenis pakaian Fashion-MNIST sekaligus, dan frekuensi tiap kelas dibuat *long-tailed*.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±10 menit

**Isi notebook:**
- Keluaran sigmoid per label + *binary cross-entropy* (bukan softmax).
- Rata-rata **makro** vs **mikro** vs **berbobot**, dan bagaimana ketidakseimbangan memisahkan ketiganya.
- Metrik khas multilabel: *subset accuracy*, *Hamming loss*, AUROC/AUPRC per kelas.
- Ambang per kelas yang dipilih di data validasi.
- BCE vs *focal loss* dengan penyeimbangan kelas; selang kepercayaan *bootstrap* untuk AUROC makro.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Membangun dataset mozaik multilabel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras import layers

(xf, yf), (xf_uji, yf_uji) = keras.datasets.fashion_mnist.load_data()
KELAS = ["kaus", "celana", "pullover", "gaun", "mantel", "sandal", "kemeja", "sneaker", "tas", "bot"]
RASIO = 50                                             # kelas terjarang 50x lebih jarang dari terbanyak
peluang_kelas = RASIO ** (-np.arange(10) / 9)
peluang_kelas /= peluang_kelas.sum()

def buat_mozaik(x, y, n, seed):
    rng = np.random.default_rng(seed)
    per_kelas = [np.where(y == c)[0] for c in range(10)]
    X = np.zeros((n, 56, 56, 1), "float32"); Y = np.zeros((n, 10), "float32")
    for i in range(n):
        for sel in range(4):
            if rng.random() < 0.25:                    # 25% sel dibiarkan kosong
                continue
            c = rng.choice(10, p=peluang_kelas)
            g = x[rng.choice(per_kelas[c])]
            r, k = divmod(sel, 2)
            X[i, r * 28:(r + 1) * 28, k * 28:(k + 1) * 28, 0] = g / 255.0
            Y[i, c] = 1
    return X, Y

X_latih, Y_latih = buat_mozaik(xf[:50000], yf[:50000], 30000, seed=0)
X_val, Y_val = buat_mozaik(xf[50000:], yf[50000:], 6000, seed=1)       # sumber gambar terpisah
X_uji, Y_uji = buat_mozaik(xf_uji, yf_uji, 10000, seed=2)

prevalensi = Y_uji.mean(0)
print("Rata-rata label per gambar (kardinalitas):", Y_latih.sum(1).mean().round(2))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [2, 1]})
ax[0].bar(KELAS, prevalensi); ax[0].set_ylabel("Prevalensi (uji)"); ax[0].tick_params(axis="x", rotation=45)
ax[1].imshow(X_latih[0, ..., 0], cmap="gray"); ax[1].axis("off")
ax[1].set_title(", ".join(KELAS[c] for c in np.where(Y_latih[0])[0]))
plt.show()

## 2. Model: satu sigmoid per label

In [ ]:
def bangun(loss):
    masukan = keras.Input(shape=(56, 56, 1))
    x = masukan
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    model = keras.Model(masukan, layers.Dense(10, activation="sigmoid")(x))   # BUKAN softmax
    model.compile(optimizer="adam", loss=loss,
                  metrics=[keras.metrics.AUC(multi_label=True, num_labels=10, name="auroc_makro"),
                           keras.metrics.AUC(curve="PR", multi_label=True, num_labels=10, name="auprc_makro")])
    return model

model_bce = bangun("binary_crossentropy")
model_bce.fit(X_latih, Y_latih, batch_size=128, epochs=15, validation_data=(X_val, Y_val))
P_val, P_uji = model_bce.predict(X_val, verbose=0), model_bce.predict(X_uji, verbose=0)

## 3. Makro, mikro, berbobot
- **Makro**: rata-rata metrik per kelas. Setiap kelas sama penting, sehingga peka terhadap kelas langka.
- **Mikro**: semua pasangan (sampel, label) digabung. Didominasi kelas yang sering muncul.
- **Berbobot**: rata-rata per kelas yang dibobot prevalensi.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, hamming_loss, accuracy_score

def ringkas(Y, P, ambang=0.5):
    Yp = (P >= ambang).astype(int)
    return {
        "AUROC makro": roc_auc_score(Y, P, average="macro"),
        "AUROC mikro": roc_auc_score(Y, P, average="micro"),
        "AUPRC makro": average_precision_score(Y, P, average="macro"),
        "AUPRC mikro": average_precision_score(Y, P, average="micro"),
        "AUPRC berbobot": average_precision_score(Y, P, average="weighted"),
        "F1 makro": f1_score(Y, Yp, average="macro", zero_division=0),
        "F1 mikro": f1_score(Y, Yp, average="micro", zero_division=0),
        "Subset accuracy": accuracy_score(Y, Yp),          # SEMUA label harus tepat
        "Hamming loss": hamming_loss(Y, Yp),               # proporsi label yang salah
    }

for k, v in ringkas(Y_uji, P_uji).items():
    print(f"{k:16s} {v:.3f}")

In [ ]:
auroc_k = roc_auc_score(Y_uji, P_uji, average=None)
auprc_k = average_precision_score(Y_uji, P_uji, average=None)
f1_k = f1_score(Y_uji, (P_uji >= 0.5).astype(int), average=None, zero_division=0)

posisi = np.arange(10); lebar = 0.27
plt.figure(figsize=(12, 4))
plt.bar(posisi - lebar, auroc_k, lebar, label="AUROC")
plt.bar(posisi, auprc_k, lebar, label="AUPRC")
plt.bar(posisi + lebar, f1_k, lebar, label="F1 @0,5")
plt.plot(posisi, prevalensi, "k_", ms=18, mew=2, label="prevalensi = AUPRC acak")
plt.xticks(posisi, KELAS, rotation=45); plt.legend(ncol=4); plt.title("Metrik per kelas (kiri: sering, kanan: langka)")
plt.show()

Perhatikan kelas di sebelah kanan: AUROC-nya masih tinggi, tetapi AUPRC dan F1 pada ambang 0,5 jauh lebih rendah. Melaporkan AUROC mikro saja akan menyembunyikan kegagalan pada kelas langka.

## 4. Ambang per kelas (dipilih di validasi)

In [ ]:
from sklearn.metrics import precision_recall_curve

def ambang_per_kelas(Y, P):
    hasil = np.zeros(Y.shape[1])
    for c in range(Y.shape[1]):
        pre, rec, th = precision_recall_curve(Y[:, c], P[:, c])
        f1 = 2 * pre * rec / np.clip(pre + rec, 1e-12, None)
        hasil[c] = th[np.argmax(f1[:-1])]
    return hasil

ambang = ambang_per_kelas(Y_val, P_val)
print("Ambang per kelas:", dict(zip(KELAS, ambang.round(2))))
a, b = ringkas(Y_uji, P_uji, 0.5), ringkas(Y_uji, P_uji, ambang)
for k in ["F1 makro", "F1 mikro", "Subset accuracy", "Hamming loss"]:
    print(f"{k:16s} ambang 0,5: {a[k]:.3f} -> per kelas: {b[k]:.3f}")

## 5. *Focal loss* dengan penyeimbangan kelas

In [ ]:
model_fokal = bangun(keras.losses.BinaryFocalCrossentropy(gamma=2.0, apply_class_balancing=True, alpha=0.75))
model_fokal.fit(X_latih, Y_latih, batch_size=128, epochs=15, validation_data=(X_val, Y_val), verbose=0)
P_uji_fokal = model_fokal.predict(X_uji, verbose=0)

def bootstrap_makro(Y, P, fungsi, n=500, seed=0):
    rng = np.random.default_rng(seed)
    nilai = []
    for _ in range(n):
        i = rng.integers(0, len(Y), len(Y))
        if (Y[i].sum(0) > 0).all():
            nilai.append(fungsi(Y[i], P[i], average="macro"))
    return np.percentile(nilai, [2.5, 97.5])

for nama, P in [("BCE", P_uji), ("Focal", P_uji_fokal)]:
    for metrik, f in [("AUROC makro", roc_auc_score), ("AUPRC makro", average_precision_score)]:
        lo, hi = bootstrap_makro(Y_uji, P, f)
        print(f"{nama:6s} {metrik}: {f(Y_uji, P, average='macro'):.3f}  (IK 95%: {lo:.3f} – {hi:.3f})")

plt.figure(figsize=(10, 3.5))
plt.plot(KELAS, average_precision_score(Y_uji, P_uji, average=None), "o-", label="BCE")
plt.plot(KELAS, average_precision_score(Y_uji, P_uji_fokal, average=None), "s-", label="Focal")
plt.ylabel("AUPRC"); plt.xticks(rotation=45); plt.legend(); plt.title("AUPRC per kelas"); plt.show()

**Daftar periksa pelaporan multilabel** (dibahas di Bab 16):
1. Laporkan metrik **per kelas** beserta prevalensinya, bukan hanya rata-rata.
2. Sebutkan jenis rata-rata (makro/mikro/berbobot) secara eksplisit.
3. Pisahkan pemilihan ambang (validasi) dari pelaporan (uji).
4. Sertakan selang kepercayaan, terutama untuk kelas langka.
5. Bila data berasal dari individu yang sama (misalnya pengguna yang sama), bagi data **per individu**, bukan per gambar.